# WO-03 Q6 — where the point sits
For a point: the L6 basin containing it, the distance from the point to the core's outline and to the nearest neighbouring basin (a divide; the outline of a coastal basin also includes the coast), the core's size measure (square root of its area, km), the nearest neighbour and how many lie within 5 and 25 km; for points outside every L6 basin, the nearest basin and its distance (input to the snap tolerance). Place sets: the 10 test places, the 258 WH Cities (`gaz.wh_cities`), the 306 WO-02 sample points (basin interior points, so their boundary distances reflect how the point was chosen), and a random 3,000 of the 27,699 `gaz.geonames_cities` for how often a settlement lies outside every L6 basin. Neighbours are by `ST_Touches`, as in the ring. Read-only.

In [1]:
%matplotlib inline
# Cell 1 — Setup: the four place sets
import sys, time, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.whg import wo03_common as C

conn = db_connect()
conn.execute("SET statement_timeout = 120000")
CITIES = conn.execute("SELECT id, city || ', ' || country, region, ST_Y(geom::geometry), ST_X(geom::geometry) FROM gaz.wh_cities ORDER BY id").fetchall()
GEO = conn.execute("SELECT gnid, name || ', ' || ccode, ccode, lat, lon FROM gaz.geonames_cities ORDER BY md5(gnid::text) LIMIT 3000").fetchall()
SAMPLE = C.population_points()
print('commit', C.commit(), '| test places:', len(C.PLACES), '| WH Cities:', len(CITIES), '| sample points:', len(SAMPLE), '| GeoNames subset:', len(GEO))

commit c6e60ac | test places: 10 | WH Cities: 258 | sample points: 306 | GeoNames subset: 3000


In [2]:
# Cell 2 — Query functions
IN_SQL = """
WITH p AS (SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326) AS g)
SELECT b.hybas_id::bigint, b.sub_area::float8, ST_Distance(p.g::geography, ST_Boundary(b.geom)::geography) / 1000.0
FROM public.basin06 b, p WHERE ST_Intersects(b.geom, p.g) LIMIT 1
"""
NB_SQL = """
SELECT n.hybas_id::bigint, ST_Distance(n.geom::geography, ST_SetSRID(ST_MakePoint(%s, %s), 4326)::geography) / 1000.0 AS km
FROM public.basin06 n JOIN public.basin06 c ON c.hybas_id = %s AND n.hybas_id <> c.hybas_id AND ST_Touches(n.geom, c.geom)
ORDER BY km
"""
NEAR_SQL = """
WITH p AS (SELECT ST_SetSRID(ST_MakePoint(%s, %s), 4326) AS g)
SELECT hybas_id, km FROM (
  SELECT b.hybas_id::bigint AS hybas_id, ST_Distance(b.geom::geography, p.g::geography) / 1000.0 AS km
  FROM public.basin06 b, p ORDER BY b.geom <-> p.g LIMIT 5) t
ORDER BY km LIMIT 1
"""
def facts(setname, pid, label, group, lat, lon, full=True):
    d = dict(set=setname, id=pid, label=label, group=group, lat=lat, lon=lon, hybas=np.nan, area_km2=np.nan, side_km=np.nan,
             d_boundary_km=np.nan, share=np.nan, nn_id=np.nan, nn_km=np.nan, n_nb=np.nan, n_nb_5km=np.nan, n_nb_25km=np.nan,
             near_basin_id=np.nan, near_basin_km=np.nan)
    r = conn.execute(IN_SQL, (lon, lat)).fetchone()
    if r is None:
        n = conn.execute(NEAR_SQL, (lon, lat)).fetchone()
        d['near_basin_id'], d['near_basin_km'] = n[0], n[1]
        return d
    d['hybas'], d['area_km2'] = r[0], r[1]
    d['side_km'] = float(np.sqrt(r[1])); d['d_boundary_km'] = r[2]; d['share'] = r[2] / d['side_km']
    if full:
        nb = conn.execute(NB_SQL, (lon, lat, r[0])).fetchall()
        d['n_nb'] = len(nb)
        if nb:
            d['nn_id'], d['nn_km'] = nb[0][0], nb[0][1]
        d['n_nb_5km'] = sum(1 for x in nb if x[1] <= 5); d['n_nb_25km'] = sum(1 for x in nb if x[1] <= 25)
    return d
print('functions defined')

functions defined


In [3]:
# Cell 3 — Test places, WH Cities and sample points; saved to a CSV
rows, t_start = [], time.perf_counter()
for name, (lat, lon) in C.PLACES.items():
    rows.append(facts('test', 0, name, 'test', lat, lon))
for k, c in enumerate(CITIES, 1):
    rows.append(facts('wh_cities', int(c[0]), c[1], c[2], float(c[3]), float(c[4])))
    if k % 100 == 0:
        print(f'  cities {k} of {len(CITIES)} in {time.perf_counter() - t_start:.0f}s')
for k, s in enumerate(SAMPLE, 1):
    rows.append(facts('sample', int(s['hybas_id_l6']), str(s['hybas_id_l6']), s['climate_zone'], float(s['lat']), float(s['lon'])))
    if k % 100 == 0:
        print(f'  sample {k} of {len(SAMPLE)} in {time.perf_counter() - t_start:.0f}s')
R6 = pd.DataFrame(rows)
for c in ['hybas', 'nn_id', 'near_basin_id', 'n_nb', 'n_nb_5km', 'n_nb_25km']:
    R6[c] = R6[c].astype('Int64')
R6.to_csv(C.OUT / 'q6_points.csv', index=False)
print(f'rows saved: {len(R6)} | this run took {time.perf_counter() - t_start:.0f}s')

  sample 300 of 306 in 4s
rows saved: 574 | this run took 4s


In [4]:
# Cell 4 — The ten test places
d = R6[R6['set'] == 'test'][['label', 'hybas', 'area_km2', 'd_boundary_km', 'nn_km', 'share', 'nn_id', 'n_nb', 'n_nb_5km', 'n_nb_25km', 'near_basin_km']].copy()
for c in ['area_km2', 'd_boundary_km', 'nn_km', 'near_basin_km']:
    d[c] = d[c].astype('float64').round(1)
d['share'] = d['share'].astype('float64').round(3)
print(d.to_string(index=False))

           label      hybas  area_km2  d_boundary_km  nn_km  share      nn_id  n_nb  n_nb_5km  n_nb_25km  near_basin_km
        Timbuktu 1060551560    3826.4            7.4    7.4  0.119 1060564740     5         0          3            NaN
         Tbilisi 2060616700   23250.8           21.6   21.6  0.142 2060616770     9         0          2            NaN
        Santa Fe 7060674070   14659.0           12.5   12.5  0.103 7060690100     7         0          1            NaN
         Kaifeng 4060675920   15266.6            6.0    6.0  0.049 4060579370     7         0          2            NaN
            Lviv 2060376210   16335.7            1.9    1.9  0.015 2060444020    12         1          2            NaN
         Poltava 2060459130   14511.5           18.0   18.0  0.150 2060454640     4         0          1            NaN
    Chaco Canyon 7060618200   11887.9           28.3   28.3  0.259 7060619370     7         0          0            NaN
      Alexandria 1060034170   16374.7   

In [5]:
# Cell 5 — Distributions: WH Cities and the sample (points inside a basin)
q = lambda a, p: float(np.nanpercentile(a, p))
L = []
for setname, note in [('wh_cities', 'WH Cities'), ('sample', 'sample (interior points; reflects the choice of point)')]:
    d = R6[(R6['set'] == setname) & R6['hybas'].notna()]
    n = len(d); out = R6[(R6['set'] == setname) & R6['hybas'].isna()]
    db = d['d_boundary_km'].astype('float64').to_numpy(); nn = d['nn_km'].astype('float64').to_numpy(); sh = d['share'].astype('float64').to_numpy()
    L.append(f'{note}: {n} inside a basin, {len(out)} outside every L6 basin')
    L.append(f'  distance to the core outline, km: 10% {q(db,10):.1f} | 50% {q(db,50):.1f} | 90% {q(db,90):.1f}')
    L.append(f'  as a share of the core size measure: 10% {q(sh,10):.3f} | 50% {q(sh,50):.3f} | 90% {q(sh,90):.3f}')
    L.append(f'  distance to the nearest neighbouring basin, km (no neighbours: {int(np.isnan(nn).sum())}): 10% {q(nn,10):.1f} | 50% {q(nn,50):.1f} | 90% {q(nn,90):.1f}')
    L.append('  nearest neighbour within: ' + ' | '.join(f'{km} km {int((nn <= km).sum())} ({100 * (nn <= km).sum() / n:.0f}%)' for km in (1, 5, 10, 25)))
    L.append('  share of the core size measure at or under: ' + ' | '.join(f'{x} {int((sh <= x).sum())} ({100 * (sh <= x).sum() / n:.0f}%)' for x in (0.02, 0.05, 0.1)))
    L.append('  nearest outline is not a divide (outline closer than the nearest neighbour by over 0.01 km): ' + str(int((db < nn - 0.01).sum() + np.isnan(nn).sum())))
    L.append('  neighbours within 5 km of the point: ' + str(d['n_nb_5km'].astype('float64').value_counts().sort_index().astype(int).to_dict()))
    L.append('')
print('\n'.join(L))

WH Cities: 254 inside a basin, 4 outside every L6 basin
  distance to the core outline, km: 10% 0.6 | 50% 7.6 | 90% 29.5
  as a share of the core size measure: 10% 0.007 | 50% 0.076 | 90% 0.223
  distance to the nearest neighbouring basin, km (no neighbours: 14): 10% 2.2 | 50% 13.5 | 90% 41.6
  nearest neighbour within: 1 km 13 (5%) | 5 km 47 (19%) | 10 km 86 (34%) | 25 km 175 (69%)
  share of the core size measure at or under: 0.02 66 (26%) | 0.05 100 (39%) | 0.1 154 (61%)
  nearest outline is not a divide (outline closer than the nearest neighbour by over 0.01 km): 84
  neighbours within 5 km of the point: {0.0: 207, 1.0: 35, 2.0: 12}

sample (interior points; reflects the choice of point): 306 inside a basin, 0 outside every L6 basin
  distance to the core outline, km: 10% 3.2 | 50% 15.0 | 90% 36.8
  as a share of the core size measure: 10% 0.084 | 50% 0.229 | 90% 0.333
  distance to the nearest neighbouring basin, km (no neighbours: 8): 10% 3.6 | 50% 16.2 | 90% 37.5
  nearest neigh

In [6]:
# Cell 6 — Points outside every L6 basin: WH Cities, test places, and a 3,000 GeoNames subset
t0 = time.perf_counter(); rows = []
for k, g in enumerate(GEO, 1):
    rows.append(facts('geonames', int(g[0]), g[1], g[2], float(g[3]), float(g[4]), full=False))
    if k % 1000 == 0:
        print(f'  geonames {k} of {len(GEO)} in {time.perf_counter() - t0:.0f}s')
GN = pd.DataFrame(rows)
GN.to_csv(C.OUT / 'q6_geonames_subset.csv', index=False)
out = GN[GN['hybas'].isna()]
nb_km = out['near_basin_km'].astype('float64').to_numpy()
L = [f'GeoNames subset: {len(GN)} points | outside every L6 basin: {len(out)} ({100 * len(out) / len(GN):.1f}%)']
if len(out):
    L.append('  distance to the nearest basin, km: ' + ' | '.join(f'{p}% {np.percentile(nb_km, p):.2f}' for p in (10, 50, 90)) + f' | max {nb_km.max():.2f}')
    L.append('  within: ' + ' | '.join(f'{km} km {int((nb_km <= km).sum())}' for km in (0.1, 0.5, 1, 5, 10, 25)))
    L.append('  the ten farthest: ' + '; '.join(f"{r.label} {r.near_basin_km:.1f} km" for r in out.sort_values('near_basin_km', ascending=False).head(10).itertuples()))
wh = R6[(R6['set'].isin(['wh_cities', 'test'])) & R6['hybas'].isna()][['set', 'label', 'near_basin_id', 'near_basin_km']].copy()
wh['near_basin_km'] = wh['near_basin_km'].astype('float64').round(2)
L.append('')
L.append('WH Cities and test places outside every L6 basin:')
L.append(wh.to_string(index=False))
print('\n'.join(L))

  geonames 3000 of 3000 in 2s
GeoNames subset: 3000 points | outside every L6 basin: 6 (0.2%)
  distance to the nearest basin, km: 10% 285.27 | 50% 2139.00 | 90% 5413.39 | max 7196.36
  within: 0.1 km 0 | 0.5 km 1 | 1 km 1 | 5 km 1 | 10 km 1 | 25 km 1
  the ten farthest: Apia, WS 7196.4 km; East Honolulu, US 3630.4 km; Kaneohe, US 3616.6 km; Assomada, CV 661.4 km; Bambous, MU 570.3 km; Akureyri, IS 0.2 km

WH Cities and test places outside every L6 basin:
      set                       label  near_basin_id  near_basin_km
     test            Tristan da Cunha     1060015860        2803.67
wh_cities    Cidade Velha, Cape Verde     1060027110         652.31
wh_cities                Levuka, Fiji     5060081550           0.00
wh_cities Angra do Heroísmo, Portugal     1060034610        1107.91
wh_cities       St. George's, Bermuda     7060042750        1052.13
